# SLEAP → DLCAnalyzer formatting

Merges the `animal` and `geom` coordinate exports for one assay/cohort and writes
DLCAnalyzer-readable CSVs into `formatted/<PHASE>/`.

**The logic lives in [`sleap_dataform.py`](sleap_dataform.py), not in this notebook.**
This notebook only sets the parameters and calls it. That matters: a notebook left open
in an editor keeps its own copy of every cell, so running it can execute a version of
the code that no longer exists on disk — which is exactly what happened after the fix
below. With the work in a module, the notebook reloads it from disk on every run and
cannot go stale.

You can skip the notebook entirely:

```bash
python sleap_dataform.py --batch B6 --assay SocP --dry-run
python sleap_dataform.py --batch B6 --assay SocP --overwrite
```

---

## What was wrong

Output names were derived by counting underscores:

```python
new_file_name = file_name.split("_", 7)[-1] + "_formatted.csv"
```

On `B6_A3R3_S1.merged_locs` that splits to `['B6','A3R3','S1.merged','locs']` and takes
`'locs'`, so **every file was written as `locs_formatted.csv`, each overwriting the
last**. The count was correct for a longer filename pattern used at some point and
silently wrong afterwards.

| Problem | Fix |
|---|---|
| `split("_", 7)` → one name for every file | `<CODE>_<PHASE>` parsed by regex; no underscore counting |
| 19 files silently collapsed into 1 | Paths planned up front; a collision or existing file aborts unless `--overwrite` |
| Output written flat, then moved by hand | Written straight into `formatted/<PHASE>/` |
| Magic numbers `+20` SocP / `+18` OFT / `+26` EPM | Bodypart count read from the header |
| `animal`/`geom` paired by `str.replace` | Paired on the parsed key; unmatched files listed |
| Row counts never checked | Asserted equal, and non-empty, before merging |
| Header written with `\n`, pandas body with `\r\n` | One explicit `LINE_TERM` for both |
| ~650 MB of intermediates written to the share and read straight back — one output came out at 1 kB instead of 12 MB | Merge done **in memory**; intermediates only with `--keep-merged` |
| A duplicated recording could pass unnoticed | Verification compares phases per animal; `nan`-safe, and an incomparable pair is reported rather than treated as clean |

Output is byte-identical to the existing corpus: validated on B6/SocP, 54 of 57 files
match exactly. The 3 that differ are D3U7 HAB/S1/S2, correctly, because those videos
were replaced and re-extracted.

## 1. Load the module

`importlib.reload` is deliberate: without it, editing `sleap_dataform.py` has no effect
until the kernel is restarted, which is the same stale-code trap in a different place.

In [ ]:
import importlib
import sys
from pathlib import Path

# Make the module importable no matter where the kernel was started.
HERE = Path.cwd() if (Path.cwd() / "sleap_dataform.py").exists() else Path(
    r"C:\Users\topohl\Documents\GitHub\SLEAPanalyzer\01_SLEAPcoords")
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import sleap_dataform as sd
importlib.reload(sd)          # pick up edits without restarting the kernel

print(f"loaded {sd.__file__}")
print(f"  assays  : {sorted(sd.PHASES)}")
print(f"  root    : {sd.BEHAVIOR_ROOT}")

## 2. Dry run

Reports what a real run would write and changes nothing. Check the pairing counts and
that no file is listed as an unexpected orphan before going further.

In [ ]:
BATCH = "B6"
ASSAY = "SocP"          # SocP | NOR | EPM | OFT

_ = sd.run(BATCH, ASSAY, dry_run=True)

## 3. Write

`overwrite=True` is required if `formatted/` is already populated. The run reports
per-file bodypart and frame counts, then verifies the output: every file distinct, and
each animal's phases compared against each other to catch a duplicated recording.

In [ ]:
report = sd.run(BATCH, ASSAY, overwrite=True)
report